In [ ]:
# agentic_rag_simplified.py
import os
from typing import List, TypedDict
from pinecone import Pinecone
from langgraph.graph import StateGraph, END
from langchain_google_vertexai import VertexAIEmbeddings, ChatVertexAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field

# --- Configuration ---
PINECONE_INDEX_NAME = "agentic-rag-kb"

# Embedding and LLM Models
EMBEDDING_MODEL_NAME = "textembedding-gecko@003"
LLM_MODEL_NAME = "gemini-2.5-flash"

# --- State Definition ---
# This class defines the "state" that flows through the graph.
class GraphState(TypedDict):
    question: str
    documents: List[dict]
    generation: str
    critique: str
    refined_generation: str


# --- Node 1: Retriever Node ---
def retrieve_kb(state: GraphState) -> GraphState:
    """
    Retrieves documents from the vector database based on the user's question.
    """
    print("---NODE: RETRIEVE KB---")
    question = state["question"]
    
    # Generate embedding for the question
    question_embedding = embeddings.embed_query(question)
    
    # Query Pinecone for top 5 similar documents
    retrieval_results = pinecone_index.query(
        vector=question_embedding,
        top_k=5,
        include_metadata=True
    )
    
    # Format the retrieved documents
    retrieved_docs = [
        {"id": match['id'], "text": match['metadata']['text']} 
        for match in retrieval_results['matches']
    ]
    
    print(f"Retrieved {len(retrieved_docs)} documents.")
    return {"documents": retrieved_docs, "question": question}

# --- Node 2: LLM Answer Node ---
def generate_answer(state: GraphState) -> GraphState:
    """
    Generates an answer using the LLM based on the retrieved documents.
    """
    print("---NODE: GENERATE INITIAL ANSWER---")
    question = state["question"]
    documents = state["documents"]
    
    # Format documents into a string context
    context = "\n\n".join([f"Citation [KB{doc['id']}]: {doc['text']}" for doc in documents])
    
    # Create a prompt for the LLM
    prompt_template = """You are an expert assistant. Answer the user's question based *only* on the following context.
    Cite your sources using the format [KBxxx] at the end of each sentence that uses that source.
    If the context is insufficient, state that you cannot answer the question.

    CONTEXT:
    {context}

    QUESTION:
    {question}

    ANSWER:"""
    prompt = ChatPromptTemplate.from_template(prompt_template)
    
    # Chain the prompt with the LLM
    rag_chain = prompt | llm
    
    # Invoke the chain to get the generation
    generation = rag_chain.invoke({"context": context, "question": question}).content
    
    print("Generated initial answer.")
    return {"generation": generation}

# --- Node 3: Self-Critique Node ---
def critique_answer(state: GraphState) -> GraphState:
    """
    Critiques the generated answer for completeness against the documents.
    """
    print("---NODE: CRITIQUE ANSWER---")
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]
    
    context = "\n\n".join([f"Citation [KB{doc['id']}]: {doc['text']}" for doc in documents])
    
    # Prompt for the critique model
    critique_prompt_template = """You are a quality assurance expert. Your task is to critique a generated answer based on a given question and context.
    
    First, review the original question and the provided context documents.
    Then, review the generated answer.
    
    Decide if the answer completely addresses the question using *all relevant information* from the context.
    - If the answer is complete and well-supported by the context, respond with only the word: COMPLETE
    - If the answer is missing key information from the context or misinterprets it, respond with: REFINE: <comma-separated keywords of missing information>
    
    Example:
    If the context mentions API versioning strategies like 'semantic versioning' and 'URL path versioning', but the answer only mentions 'URL path versioning', a correct response would be:
    REFINE: semantic versioning
    
    CONTEXT:
    {context}
    
    QUESTION:
    {question}
    
    GENERATED ANSWER:
    {generation}
    
    YOUR CRITIQUE:"""
    prompt = ChatPromptTemplate.from_template(critique_prompt_template)
    
    critique_chain = prompt | llm
    critique = critique_chain.invoke({
        "context": context, 
        "question": question, 
        "generation": generation
    }).content
    
    print(f"Critique result: {critique}")
    return {"critique": critique}

# --- Node 4: Refinement Node ---
def refine_answer(state: GraphState) -> GraphState:
    """
    Refines the answer by retrieving one more document and regenerating.
    """
    print("---NODE: REFINE ANSWER---")
    question = state["question"]
    existing_documents = state["documents"]
    critique = state["critique"]
    
    # Extract missing keywords from the critique
    missing_keywords = critique.replace("REFINE:", "").strip()
    
    # Augment the original question with these keywords for a more targeted search
    refined_query = f"{question} {missing_keywords}"
    print(f"Refined query for new retrieval: {refined_query}")
    
    # Get a new embedding for the refined query
    refined_embedding = embeddings.embed_query(refined_query)
    
    # Retrieve one more document, excluding IDs we already have
    existing_ids = [doc['id'] for doc in existing_documents]
    retrieval_results = pinecone_index.query(
        vector=refined_embedding,
        top_k=2, # Fetch 2 in case the top one is already included
        include_metadata=True
    )
    
    # Find the first new document
    new_doc = None
    for match in retrieval_results['matches']:
        if match['id'] not in existing_ids:
            new_doc = {"id": match['id'], "text": match['metadata']['text']}
            break
            
    if not new_doc:
        print("Could not find a new, relevant document. Using original answer.")
        return {"refined_generation": state["generation"]} # Fallback

    print(f"Retrieved one additional document: [KB{new_doc['id']}]")
    all_documents = existing_documents + [new_doc]

    # Use the same generation logic with the updated document set
    refined_state = {"question": question, "documents": all_documents}
    final_generation_map = generate_answer(refined_state)
    
    return {"refined_generation": final_generation_map["generation"]}

# --- Conditional Edge Logic ---
def should_refine(state: GraphState) -> str:
    """
    Determines the next step based on the critique.
    """
    print("---DECISION: SHOULD REFINE?---")
    critique = state["critique"]
    if critique.startswith("REFINE:"):
        print("Decision: Yes, refinement needed.")
        return "refine"
    else:
        print("Decision: No, answer is complete.")
        return "end"

# --- Build the Graph ---
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("retrieve_kb", retrieve_kb)
workflow.add_node("generate_answer", generate_answer)
workflow.add_node("critique_answer", critique_answer)
workflow.add_node("refine_answer", refine_answer)

# Set entry and edges
workflow.set_entry_point("retrieve_kb")
workflow.add_edge("retrieve_kb", "generate_answer")
workflow.add_edge("generate_answer", "critique_answer")
workflow.add_conditional_edges(
    "critique_answer",
    should_refine,
    {
        "refine": "refine_answer",
        "end": END,
    },
)
workflow.add_edge("refine_answer", END)

# Compile the graph
app = workflow.compile()

# --- Testing Queries ---
def run_query(query: str):
    """Helper function to run a query and print the final result."""
    print(f"\n{'='*50}\n🚀 EXECUTING QUERY: {query}\n{'='*50}")
    
    inputs = {"question": query}
    final_state = None
    # The `stream` method yields the state after each node execution
    for output in app.stream(inputs, stream_mode="values"):
        final_state = output

    print("\n---FINAL RESULT---")
    final_answer = final_state.get("refined_generation") or final_state.get("generation")
    print(final_answer)
    print(f"{'='*50}\n")


if __name__ == "__main__":
    test_queries = [
        "What are best practices for caching?",
        "How should I set up CI/CD pipelines?",
        "What are performance tuning tips?",
        "How do I version my APIs?",
        "What should I consider for error handling?",
    ]
    
    for q in test_queries:
        run_query(q)